In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.chat_history import InMemoryChatMessageHistory # method -- overide ( you define by yourself )
# add_message, clear_message
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

True

In [13]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

history = InMemoryChatMessageHistory()
history.messages


# add message
# history.add_user_message("I have a billing complaint from John about invoice #1042.")
# history.add_ai_message("Understood. This is classified as Billing — High Priority.")


[]

In [9]:
history

InMemoryChatMessageHistory(messages=[HumanMessage(content='I have a billing complaint from John about invoice #1042.', additional_kwargs={}, response_metadata={}), AIMessage(content='Understood. This is classified as Billing — High Priority.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])

In [10]:
# history ==> prompt ==> MessagesPlaceholder
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [12]:
prompt = ChatPromptTemplate.from_messages([
    ("system", """\
You are a professional email support agent.
Classify complaints as: Billing / Technical / General.
Priority: High (financial impact > $100 or urgent) / Medium / Low.
Remember all customer details throughout the conversation."""),
    MessagesPlaceholder(variable_name="history"),  # ← past turns injected here
    ("human", "{input}"),                           # ← new message here
])

chain = prompt | llm | StrOutputParser()

In [15]:
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
   
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


In [16]:
from langchain_core.runnables.history import RunnableWithMessageHistory


email_assistant = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",      # matches {"input": "..."} in invoke()
    history_messages_key="history",  # matches MessagesPlaceholder("history")
)

In [17]:
cfg = {"configurable": {"session_id": "support_john_001"}}

In [18]:
r1 = email_assistant.invoke(
    {"input": "I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250."},
    config=cfg
)

In [19]:
store

{'support_john_001': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \n- Name: John  \n- Invoice Number: #1042  \n- Charged Amount: $500  \n- Expected Amount: $250  \n\nPlease let me know how you would like to proceed with this complaint.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}

In [22]:
r2 = email_assistant.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg
)


In [23]:
r2

"John's complaint falls under the category of **Billing**."

In [24]:
store

{'support_john_001': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \n- Name: John  \n- Invoice Number: #1042  \n- Charged Amount: $500  \n- Expected Amount: $250  \n\nPlease let me know how you would like to proceed with this complaint.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint falls under the category of **Billing**.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complain

In [25]:
cfg_sarah = {"configurable": {"session_id": "session_sarah"}}
r2 = email_assistant.invoke(
    {"input": "What category does his complaint fall under?"},
    config=cfg_sarah
)

r2


'Please provide the details of the complaint so I can classify it accordingly.'

In [26]:
store

{'support_john_001': InMemoryChatMessageHistory(messages=[HumanMessage(content='I received a billing complaint from John about invoice #1042. He was charged $500 instead of $250.', additional_kwargs={}, response_metadata={}), AIMessage(content='Complaint Classification: Billing  \nPriority: High  \n\nCustomer Details:  \n- Name: John  \n- Invoice Number: #1042  \n- Charged Amount: $500  \n- Expected Amount: $250  \n\nPlease let me know how you would like to proceed with this complaint.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complaint falls under the category of **Billing**.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What category does his complaint fall under?', additional_kwargs={}, response_metadata={}), AIMessage(content="John's complain

In [ ]:
# EXPERIMENT: What happens when you use a different session_id?
# Try sending the same follow-up with a brand new session_id
# The model will have no context!

# login to application ==> token id
# store ==token id
# hi
# hello
# store in my db ==> id ={}
# what was my alst question ==>
